# Phase 10: Inventory Decision Engine & Discrete-Event Simulation
## MarketPulse: E-Commerce Demand Forecasting Engine

Demand forecasting is only valuable when translated into physical supply chain decisions. In this notebook, we bridge the gap between machine learning forecasts and operational replenishment by simulating:

1. **Mathematical Safety Stock ($SS$) & Reorder Point ($ROP$)**: Incorporating demand variance, lead-time stochasticity, and target service levels.
2. **Wilson Economic Order Quantity ($EOQ$) & $(s, S)$ Policy Optimization**: Balancing fixed ordering costs against working capital holding costs.
3. **Discrete-Event Monte Carlo Simulation**: Tracking physical inventory variables ($I_t^{OH}, I_t^{OO}, B_t, IP_t$) across daily customer transactions.
4. **Supply Chain KPI Suite**: Measuring Fill Rate, Cycle Service Level (CSL), Inventory Turnover, Days of Inventory Outstanding (DIO), and Total Cost of Inventory (TCOI).
5. **Service Level vs. Working Capital Pareto Frontier**: Quantifying the non-linear trade-off between customer availability and inventory investment.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.hierarchy.hierarchy_builder import HierarchyTree
from src.hierarchy.aggregation import HierarchicalAggregator
from src.inventory.decision_engine import InventoryDecisionEngine, InventoryPolicyParams
from src.inventory.simulator import InventorySimulator, SimulationConfig

# Load M5 data
data_path = PROJECT_ROOT / 'data' / 'processed' / 'm5_processed.parquet'
df = pd.read_parquet(data_path)
tree = HierarchyTree(metadata_df=df)
aggregator = HierarchicalAggregator(hierarchy=tree.m5_hierarchy)
bottom_wide = aggregator.prepare_bottom_wide_dataframe(df)
dates = pd.date_range(start=bottom_wide.index[0], periods=len(bottom_wide), freq='D')
bottom_wide.index = dates

print(f"Loaded portfolio: {bottom_wide.shape[1]} SKUs over {len(bottom_wide)} days.")

--- 
## 1. Computing Optimal Replenishment Policies via InventoryDecisionEngine
We parameterize lead-time distributions ($\mu_L = 3.0$ days, $\sigma_L = 1.0$ days), cost structures, and target service levels ($95\%$ CSL).

In [ ]:
policy_params = InventoryPolicyParams(
    lead_time_mean=3.0,
    lead_time_std=1.0,
    service_level=0.95,
    unit_cost=10.0,
    annual_holding_rate=0.25,
    order_cost=25.0,
    stockout_penalty=15.0,
)
engine = InventoryDecisionEngine(params=policy_params)

decisions = {}
policy_records = []
for col in bottom_wide.columns:
    s = bottom_wide[col]
    dec = engine.compute_policy(sku_id=str(col), demand_mean=float(s.mean()), demand_std=float(s.std()))
    decisions[str(col)] = dec
    policy_records.append(dec.to_dict())

df_policies = pd.DataFrame(policy_records)
display(df_policies[['sku_id', 'safety_stock', 'reorder_point', 'order_quantity', 'order_up_to_level', 'estimated_total_annual_cost']].head(10))

--- 
## 2. Daily Inventory Trajectory Simulation
We visualize the step-by-step physical inventory dynamics for a representative SKU under continuous $(s, S)$ review, highlighting stock arrivals, sales fulfillment, and inventory position.

In [ ]:
rep_sku = bottom_wide.columns[0]
simulator = InventorySimulator(config=SimulationConfig(policy_type='(s, S)', stochastic_lead_time=True, random_seed=42))
kpis, df_log = simulator.simulate_series(demand=bottom_wide[rep_sku], decision=decisions[str(rep_sku)], policy_params=policy_params)

print(f"Simulation KPIs for {rep_sku}:")
print(f"  Fill Rate: {kpis.fill_rate_pct:.2f}% | CSL: {kpis.cycle_service_level_pct:.2f}%")
print(f"  Average On-Hand: {kpis.average_on_hand:.2f} units | Turnover: {kpis.annualized_inventory_turnover:.2f}x")
print(f"  Total Cost: ${kpis.total_cost_of_inventory:.2f}")

# Trajectory Plot
fig, ax = plt.subplots(figsize=(12, 5))
steps = df_log['step']
ax.plot(steps, df_log['ending_on_hand'], label='On-Hand Inventory (Physical Stock)', color='#2b5c8f', linewidth=2)
ax.plot(steps, df_log['inventory_position'], label='Inventory Position (On-Hand + On-Order)', color='#2ca02c', linestyle='--', alpha=0.8)
ax.axhline(decisions[str(rep_sku)].reorder_point, color='red', linestyle=':', label=f"Reorder Point s = {decisions[str(rep_sku)].reorder_point:.1f}")
ax.axhline(decisions[str(rep_sku)].order_up_to_level, color='purple', linestyle=':', label=f"Order-Up-To Level S = {decisions[str(rep_sku)].order_up_to_level:.1f}")
ax.bar(steps, df_log['demand'], width=0.4, label='Daily Customer Demand', color='#e74c3c', alpha=0.5)
ax.set_xlabel('Simulation Day')
ax.set_ylabel('Units')
ax.set_title(f'Discrete-Event Inventory Trajectory under (s, S) Replenishment: {rep_sku}')
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

--- 
## 3. Policy Benchmark: (s, S) Order-Up-To vs. (s, Q) Fixed Lot EOQ
We compare the portfolio-wide performance and cost breakdowns between the adaptive $(s, S)$ policy and the rigid $(s, Q)$ fixed order quantity policy.

In [ ]:
kpi_df = pd.read_csv(PROJECT_ROOT / 'experiments' / 'results' / 'inventory_simulation_kpis.csv')

# Compare cost distributions
summary = kpi_df.groupby('policy_evaluated')[['fill_rate_pct', 'cycle_service_level_pct', 'average_on_hand', 'annualized_inventory_turnover', 'total_cost_of_inventory']].mean()
display(summary)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
kpi_df.boxplot(column='fill_rate_pct', by='policy_evaluated', ax=ax1, patch_artist=True)
ax1.set_title('Fill Rate Distribution (%)')
ax1.set_ylabel('Fill Rate (%)')

kpi_df.boxplot(column='total_cost_of_inventory', by='policy_evaluated', ax=ax2, patch_artist=True)
ax2.set_title('Total Cost of Inventory Distribution ($)')
ax2.set_ylabel('Cost ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()

--- 
## 4. Service Level vs. Working Capital Efficient Frontier
We trace out the Pareto efficient curve by sweeping target service levels from $80\%$ to $99\%$, exposing the non-linear holding cost inflection point.

In [ ]:
frontier_df = pd.read_csv(PROJECT_ROOT / 'experiments' / 'results' / 'inventory_tradeoff_frontier.csv')
display(frontier_df)

fig, ax1 = plt.subplots(figsize=(9, 4.5))
color = 'tab:blue'
ax1.plot(frontier_df['Target CSL (%)'], frontier_df['Average Inventory Value ($)'], marker='o', color=color, linewidth=2, label='Working Capital ($)')
ax1.set_xlabel('Target Cycle Service Level (%)')
ax1.set_ylabel('Average Inventory Value ($)', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.plot(frontier_df['Target CSL (%)'], frontier_df['Total Cost of Inventory ($)'], marker='s', color=color, linewidth=2, linestyle='--', label='TCOI ($)')
ax2.set_ylabel('Total Cost of Inventory ($)', color=color)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Pareto Frontier: Working Capital & Cost vs. Target Service Level')
plt.tight_layout()
plt.show()

## Summary & Operational Directives
1. **Continuous Replenishment**: The $(s, S)$ Order-Up-To rule adapts to multi-unit demand surges, eliminating inventory deficits faster than fixed-lot $EOQ$.
2. **Exponential Frontier**: Moving service level from $95\%$ to $99\%$ requires steep increases in safety stock for marginal availability gains.
3. **Closed-Loop Feedback**: Downstream supply chain simulators validate forecasting models on business metrics (holding and stockout costs) rather than pure statistical errors.